# NB_lab_test_trends

Builds the **`databricks_rg.gold.lab_test_trends`** gold table — one row per test type per calendar month with abnormal/critical rates and month-over-month trend via LAG(). Full rebuild every run.

In [0]:
%sql
CREATE OR REPLACE TABLE databricks_rg.gold.lab_test_trends AS

WITH base AS (
    SELECT
        lr.test_name,
        YEAR(lr.test_date)                                               AS year,
        MONTH(lr.test_date)                                              AS month,
        c.month_name,
        CONCAT('Q', c.quarter)                                           AS quarter_name,
        COUNT(*)                                                         AS total_tests,
        COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)       AS abnormal_test_count,
        COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)       AS critical_test_count
    FROM databricks_rg.silver.lab_results      lr
    LEFT JOIN databricks_rg.silver.calendar    c
           ON c.date_key = CAST(DATE_FORMAT(DATE_TRUNC('month', lr.test_date), 'yyyyMMdd') AS INT)
    GROUP BY lr.test_name, YEAR(lr.test_date), MONTH(lr.test_date), c.month_name, CONCAT('Q', c.quarter)
)

SELECT
    test_name,
    year,
    month,
    month_name,
    quarter_name,
    total_tests,
    abnormal_test_count,
    critical_test_count,
    ROUND(abnormal_test_count * 100.0 / total_tests, 2)                  AS abnormal_rate_pct,
    ROUND(critical_test_count * 100.0 / total_tests, 2)                  AS critical_rate_pct,
    LAG(ROUND(abnormal_test_count * 100.0 / total_tests, 2)) OVER (
        PARTITION BY test_name ORDER BY year, month
    )                                                                    AS prev_month_abnormal_rate_pct,
    ROUND(
        ROUND(abnormal_test_count * 100.0 / total_tests, 2)
        - LAG(ROUND(abnormal_test_count * 100.0 / total_tests, 2)) OVER (
            PARTITION BY test_name ORDER BY year, month
          )
    , 2)                                                                 AS abnormal_rate_change_pct
FROM base

In [0]:
df_lab_trends = spark.table("databricks_rg.gold.lab_test_trends")
record_count = df_lab_trends.count()
dbutils.notebook.exit(str(record_count))